In [ ]:
import pandas as pd
import random
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel, BertConfig, BertModel
# import adapters
# from adapters import AutoAdapterModel
import gc
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import matplotlib

import pickle
import time
import memory_profiler

%load_ext memory_profiler

from pathlib import Path
import distro

%load_ext watermark

In [ ]:
%load_ext IPython.extensions.autoreload
%autoreload 2

from text_visualizations.train_stuff import (
    fix_all_seeds,
)
from text_visualizations.config_helpers import load_config

The IPython.extensions.autoreload extension is already loaded. To reload it, use:
  %reload_ext IPython.extensions.autoreload


In [ ]:
import black
import jupyter_black

jupyter_black.load(line_length=79)

In [ ]:
variables_path = Path("../results/variables")
figures_path = Path("../results/figures/tmp")
data_path = Path("../data")

In [ ]:
plt.style.use("matplotlib_style.txt")

In [ ]:
%watermark -t -d -tz -u -v -iv -w -m -h -p transformers -p openTSNE
print(distro.name(pretty=True))

Last updated: 2025-04-03 11:47:05CEST

Python implementation: CPython
Python version       : 3.12.1
IPython version      : 9.0.2

openTSNE: not installed

Compiler    : GCC 8.5.0 20210514 (Red Hat 8.5.0-21)
OS          : Linux
Release     : 4.18.0-553.el8_10.x86_64
Machine     : x86_64
Processor   : x86_64
CPU cores   : 64
Architecture: 64bit

Hostname: gber7

sklearn            : 1.6.1
matplotlib         : 3.10.1
jupyter_black      : 0.4.0
numpy              : 2.1.3
distro             : 1.9.0
transformers       : 4.50.2
memory_profiler    : 0.61.0
torch              : 2.6.0
black              : 25.1.0
pandas             : 2.2.3
pathlib            : 1.0.1
text_visualizations: 0.1.0

Watermark: 2.5.0

Rocky Linux 8.10 (Green Obsidian)


# Import

In [ ]:
%%time
iclr = pd.read_parquet(
    data_path / "iclr25v2.parquet",
    engine="fastparquet",  # "pyarrow",
)

CPU times: user 168 ms, sys: 46.6 ms, total: 215 ms
Wall time: 235 ms


In [ ]:
print(iclr.keywords.iloc[0])
print(type(iclr.keywords.iloc[0]))
print(iclr.scores.iloc[0])
print(type(iclr.scores.iloc[0]))

['deep learning', 'transfer learning']
<class 'list'>
[np.int64(6), np.int64(4), np.int64(5)]
<class 'list'>


In [ ]:
iclr.keywords = iclr.keywords.transform(lambda x: list(x))
iclr.scores = iclr.scores.transform(lambda x: list(x))

In [ ]:
print(iclr.keywords.iloc[0])
print(type(iclr.keywords.iloc[0]))
print(iclr.scores.iloc[0])
print(type(iclr.scores.iloc[0]))

['deep learning', 'transfer learning']
<class 'list'>
[np.int64(6), np.int64(4), np.int64(5)]
<class 'list'>


In [ ]:
iclr

,year,id,title,abstract,authors,decision,scores,keywords,labels
0,2017,B1-Hhnslg,Prototypical Networks for Few-shot Learning,A recent approach to few-shot classification c...,"Jake Snell, Kevin Swersky, Richard Zemel",Reject,"[6, 4, 5]","[deep learning, transfer learning]",transfer learning
1,2017,B1-q5Pqxl,Machine Comprehension Using Match-LSTM and Ans...,Machine comprehension of text is an important ...,"Shuohang Wang, Jing Jiang",Accept (Poster),"[6, 6, 7]","[natural language processing, deep learning]",language models
2,2017,B16Jem9xe,Learning in Implicit Generative Models,Generative adversarial networks (GANs) provide...,"Shakir Mohamed, Balaji Lakshminarayanan",Invite to Workshop Track,"[8, 7, 6]",[unsupervised learning],unlabeled
3,2017,B16dGcqlx,Third Person Imitation Learning,Reinforcement learning (RL) makes it possible ...,"Bradly C Stadie, Pieter Abbeel, Ilya Sutskever",Accept (Poster),"[6, 5, 6]",[],unlabeled
4,2017,B184E5qee,Improving Neural Language Models with a Contin...,We propose an extension to neural network lang...,"Edouard Grave, Armand Joulin, Nicolas Usunier",Accept (Poster),"[7, 9, 5]",[natural language processing],language models
...,...,...,...,...,...,...,...,...,...
34519,2025,zxO4WuVGns,Inverse decision-making using neural amortized...,Bayesian observer and actor models have provid...,"Dominik Straub, Tobias F. Niehues, Jan Peters,...",Accept (Poster),"[6, 6, 6]","[bayesian actor models, perception and action,...",unlabeled
34520,2025,zxbQLztmwb,Emergent Symbol-Like Number Variables in Artif...,There is an open question of what types of num...,"Satchel Grant, Noah Goodman, James Lloyd McCle...",Reject,"[3, 5, 6, 5]","[mechanistic interpretability, numeric cogniti...",unlabeled
34521,2025,zxqdVo9FjY,Generalization for Least Squares Regression wi...,Random matrix theory has proven to be a valuab...,"Jiping Li, Rishi Sonthalia",Reject,"[5, 3, 5, 5, 6]","[generalization, random matrix theory, spiked ...",unlabeled
34522,2025,zyGrziIVdE,Exploration by Running Away from the Past,The ability to explore efficiently and effecti...,"Paul-Antoine LE TOLGUENEC, Yann Besse, Florent...",Reject,"[3, 3, 5, 3]","[reinforcement learning, exploration, deep lea...",RL


# Playground

In [ ]:
hidden_dims = [1]

In [ ]:
from text_visualizations.models import ModelProjector
from text_visualizations.train_stuff import mean_pool

In [ ]:
test_model = ModelProjector(
    "sentence-transformers/all-mpnet-base-v2",
    mean_pool,
    hidden_dims=[1536, 2],
    output_dim=2,
)

In [ ]:
test_model

NameError: name 'test_model' is not defined

## Test load configs

In [ ]:
configs_path = Path("../configs")

In [ ]:
from text_visualizations.config_helpers import load_config

In [ ]:
config = load_config("exp001.yaml", configs_dir_path=configs_path)

In [ ]:
merged_config

{'model': {'model_name': 'new MODel',
  'model_path': 'sentence-transformers/all-mpnet-base-v2',
  'in_dim': 768,
  'hidden_dims': 1536,
  'output_dim': 2},
 'data_loader': {'batch_size': 64},
 'training': {'eval_every_epochs': True,
  'eval_every_batches': 0,
  'eval_rep': 'None',
  'dist_metric': 'euclidean',
  'mteb_tasks': 'None',
  'n_epochs': 10,
  'lr': '2e-5',
  'scale': 20.0}}

In [ ]:
import yaml


with open("../configs/exp001.yaml", "r") as f:
    base_config = yaml.safe_load(f)
base_config

{'model': {'model_name': 'new MODel'}, 'data_loader': None, 'training': None}

In [ ]:
base_config["model"]

{'model_name': 'new MODel'}

In [ ]:
isinstance(base_config["data_loader"], dict)

False

In [ ]:
from copy import deepcopy
from text_visualizations.config_helpers import deep_update

result = deepcopy(base_dict)

for key, value in update_dict.items():
    # If the value is a nested dictionary, recurse
    if (
        isinstance(value, dict)
        and key in result
        and isinstance(result[key], dict)
    ):
        result[key] = deep_update(result[key], value)
    elif value is not None:
        # Otherwise just update the value
        result[key] = value

In [ ]:
result

{'model': {'model_name': 'new MODel',
  'model_path': 'sentence-transformers/all-mpnet-base-v2',
  'in_dim': 768,
  'hidden_dims': 1536,
  'output_dim': 2},
 'data_loader': {'batch_size': 64},
 'training': {'eval_every_epochs': True,
  'eval_every_batches': 0,
  'eval_rep': 'None',
  'dist_metric': 'euclidean',
  'mteb_tasks': 'None',
  'n_epochs': 10,
  'lr': '2e-5',
  'scale': 20.0}}

In [ ]:
config.keys()
config

{'model': [{'model_name': 'new MODel'}], 'data_loader': None, 'training': None}

In [ ]:
config["model"]["model_name"]
config["model"]["model_path"]
config["model"]["in_dim"]
config["model"]["hidden_dims"]
config["model"]["output_dim"]

In [ ]:
config["data_loader"]["batch_size"]
config["data_loader"]["n_cons_sntcs"]


In [ ]:
config["training"]["eval_every_epochs"]
config["training"]["eval_every_batches"]
config["training"]["eval_rep"]
config["training"]["dist_metric"]
config["training"]["mteb_tasks"]
config["training"]["n_epochs"]
config["training"]["lr"]
config["training"]["scale"]

### Pooler

In [ ]:
import importlib

def get_function(function_str):
    """
    Dynamically import pooler function from a module.
    
    Args:
        function_str (str): A string like 'train_stuff.mean_pool' (first python file, second funtion name, separated by a dot) 
    """
    assert isinstance(function_str, str), "Input must be a string of the format 'train_stuff.mean_pool'"
    
    # Handle simple string case like 'poolers.mean_pooler'
    module_path, function_name = function_str.rsplit('.', 1)
    module_path = "text_visualizations." + module_path
    module = importlib.import_module(module_path)

    return getattr(module, function_name)


In [ ]:
# import text_visualizations
# import text_visualizations.train_stuff


a = get_function('train_stuff.mean_pool')

In [ ]:
a

<function text_visualizations.train_stuff.mean_pool(token_embeds, attention_mask)>

In [ ]:
from text_visualizations.train_stuff import mean_pool
b = mean_pool

In [ ]:
b

<function text_visualizations.train_stuff.mean_pool(token_embeds, attention_mask)>

### File name

In [ ]:
config_path = "exp001.yaml"

In [ ]:
import os
exp_name = os.path.basename(config_path).split('.')[0]
exp_name

'exp001'

In [ ]:
def create_results_path(exp_config_path, config_dict, variables_path):
    """
    Name example: variables_path/sbert/iclr/exp001
    
    """

    # Extract experiment name from config file path
    exp_name = os.path.basename(config_path).split('.')[0]

    results_dir = variables_path / Path(model_name / exp_name)
    results_dir.mkdir(parents=True, exist_ok=True)

    revariables_path / 
    
    # Create results directory if it doesn't exist
    results_dir = os.path.join('results', exp_name)
    os.makedirs(results_dir, exist_ok=True)
    
    # Add metadata
    merged_config['timestamp'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    merged_config['base_config'] = base_config_path
    merged_config['experiment_config'] = config_path
    
    # Save merged configuration to results directory for reproducibility
    with open(os.path.join(results_dir, 'config.yaml'), 'w') as f:
        yaml.dump(merged_config, f, default_flow_style=False, sort_keys=False)
    
    return results_dir

In [ ]:
saving_path = (
Path(config["model"]['model_name'].lower())
    / Path(config["data_loader"]['dataset'].lower())
    / Path(exp_name)
)


In [ ]:
saving_path

PosixPath('new model/iclr/exp001')

### Add losses

In [ ]:
a = np.ones((3, 10))
print(a)

[[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]]


In [ ]:
np.mean(a, axis=1).shape
np.mean(a, axis=1)

array([1., 1., 1.])

In [ ]:
np.mean(a[0])

np.float64(1.0)

In [ ]:
import time
# import sleep

start = time.time()
for i in range(2):
    time.sleep(1)
end = time.time()

print(start)
print(end)
print(end-start)

1743672733.5365684
1743672735.5372045
2.000636100769043


In [ ]:
from datetime import datetime
datetime.now().strftime("%Y-%m-%d %H:%M:%S")

'2025-04-03 11:33:06'

In [ ]:
from datetime import timedelta

def convert_seconds_using_timedelta(seconds):
    td = timedelta(seconds=seconds)
    return str(td)

# Example usage
print(convert_seconds_using_timedelta(5000))  # Output: "1:23:20"

1:23:20


In [ ]:
from datetime import timedelta



  # Output: "1:23:20"

'1:23:20'

In [ ]:
from text_visualizations.config_helpers import get_git_commit_hash
get_git_commit_hash()

'e0a8e8a6fa4091a55895616892926e2b9fde3847'